# Konu 3 — Makine öğrenmesi nedir?

Hücreleri **yukarıdan aşağı, sırayla** çalıştır: hücreye tıkla, **Shift + Enter**.
Her hücre bir öncekinin oluşturduğu değişkeni kullanır; atlarsan hata alırsın.

## Isınma — sayaç neyi sayıyor?

Bu konuda tablo hâlinde veriyle çalışacağız. Tablonun her satırı Python'da bir **sözlük** olacak.
Üç satırlık küçük bir örnek:

In [ ]:
kayitlar = [
    {"ad": "kırmızı", "etiket": "enerjik"},
    {"ad": "orta mavi", "etiket": "sakin"},
    {"ad": "şeftali", "etiket": "enerjik"},
]

ilk = kayitlar[0]
print(ilk["etiket"])

Etiketleri sayalım. `{'enerjik': 2, 'sakin': 1}` yazmalı. Ne yazıyor? Hata mesajı yok, ama sonuç yanlış.
Hatayı bul ve düzelt.

In [ ]:
sayac = {}
for kayit in kayitlar:
    etiket = kayit["ad"]
    if etiket in sayac:
        sayac[etiket] = sayac[etiket] + 1
    else:
        sayac[etiket] = 1

print(sayac)

## Adım 1 — Sınıfın verisine bak

`veri/renkler-etiketli.csv`: sınıfın Konu 2 ödevinde etiketlediği renkler.
Her satır **bir öğrencinin bir renge verdiği etiket**.

In [ ]:
import csv

with open("veri/renkler-etiketli.csv", encoding="utf-8") as dosya:
    kayitlar = list(csv.DictReader(dosya))

print(len(kayitlar))
print(kayitlar[0])

Sınıf "krem" rengini nasıl etiketlemiş? Konu 1'deki sayacın aynısı:

In [ ]:
sayac = {}
for kayit in kayitlar:
    if kayit["ad"] == "krem":
        etiket = kayit["etiket"]
        if etiket in sayac:
            sayac[etiket] = sayac[etiket] + 1
        else:
            sayac[etiket] = 1

print(sayac)

`"krem"` yerine `"kırık beyaz"` yazıp hücreyi yeniden çalıştır. İki renk ekranda neredeyse aynı.
Sınıf ikisine de aynı şeyi mi demiş?

## Adım 2 — Rengi sayıya çevir: X ve y

Model "krem" kelimesini anlamaz, sayı ister. Renk seçicide `#E63946` yazınca yanında
**R 230, G 57, B 70** görürsün: kırmızı, yeşil, mavi miktarı. Dosyada bu üç sayı hazır:
`r`, `g`, `b` sütunları. Dosyadan okunan her şey metin olduğu için `int(...)` ile sayıya çeviriyoruz.

In [ ]:
X = []
y = []
for kayit in kayitlar:
    r = int(kayit["r"])
    g = int(kayit["g"])
    b = int(kayit["b"])
    X.append([r, g, b])
    y.append(kayit["etiket"])

print(len(X), len(y))
print(X[0], y[0])

- **X**: modelin **baktığı** şey (üç sayı)
- **y**: modelin **tahmin etmesi gereken** şey (etiket)

`X[0]`'ın cevabı `y[0]`. İkisi aynı döngüde, aynı sırada doluyor.

## Adım 3 — Veriyi ikiye böl: eğitim ve test

Modeli 240 örnekle eğitip aynı 240 örnekle sınarsak ezberi ölçeriz.
Dörtte birini ayırıyoruz; model onları hiç görmeyecek.

- `train_test_split` dört liste geri verir; dördünü birden, **bu sırayla** alıyoruz:
  eğitim renkleri, test renkleri, eğitim etiketleri, test etiketleri.
- `test_size=0.25`: dörtte biri teste. `random_state=42`: satırları her seferinde **aynı**
  biçimde karıştır; böylece herkesin sonucu aynı çıkar. (42'nin özel bir anlamı yok.)

In [ ]:
from sklearn.model_selection import train_test_split

X_egitim, X_test, y_egitim, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
print(len(X_egitim), len(X_test))

## Adım 4 — Modeli eğit ve ölç

> Bu renge en çok benzeyen 5 rengi bul. Onlar ne dediyse onu de.

`fit`: "bu örneklere bak ve öğren." `score`: "hiç görmediklerinin kaçını bildin?"

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_egitim, y_egitim)
dogruluk = model.score(X_test, y_test)
print(round(dogruluk, 2))

Bu sayı iyi mi? Kıyas gerekir. Hiç düşünmeden hep "enerjik" deseydik kaçını bilirdik?

In [ ]:
enerjik = 0
for etiket in y_test:
    if etiket == "enerjik":
        enerjik = enerjik + 1

kor_tahmin = enerjik / len(y_test)
print(round(kor_tahmin, 2))

Üstten kıyas da var: sınıf aynı renge farklı etiketler verdiği için hiçbir model belli bir sayıyı
geçemez (**tavan**). Üç sayı yan yana slaytta.

Adım 4'ün ilk hücresinde `n_neighbors=5` yerine `1` ya da `15` yaz, o hücreyi yeniden çalıştır. Ne değişti?

## Adım 5 — Yanıldığı yerlere bak

Hangi renklerde yanıldı? Bunun için test satırlarının adlarını bilmemiz gerek.
`kayitlar`'ı **aynı** `random_state` ile bölünce aynı 60 satır teste düşer.

In [ ]:
egitim_kayitlari, test_kayitlari = train_test_split(
    kayitlar, test_size=0.25, random_state=42
)
print(len(test_kayitlari))

Her test satırını tek tek modele soruyoruz. `predict` her zaman bir **liste** ister, tek renk sorsak bile:
`[renk]`. Cevap da liste içinde gelir: `tahminler[0]`.

In [ ]:
for kayit in test_kayitlari:
    r = int(kayit["r"])
    g = int(kayit["g"])
    b = int(kayit["b"])
    renk = [r, g, b]
    tahminler = model.predict([renk])
    tahmin = tahminler[0]
    if tahmin != kayit["etiket"]:
        print(kayit["ad"], "öğrenci:", kayit["etiket"], "model:", tahmin)

Listeden bir renk seç (ör. `"şeftali"`). Adım 1'deki sayaç hücresinde `"krem"` yerine o rengin
adını yaz ve çalıştır. Model sınıfın çoğunluğunu mu söylüyor, yoksa azınlıktaki öğrencinin
etiketi mi "yanlış" sayılıyor?

## Adım 6 — Veri arttıkça ne oluyor?

Aynı modeli önce eğitim verisinin ilk 18 örneğiyle, sonra 45, 90, 135 ve 180 örnekle eğitiyoruz.
`X_egitim[:adet]` baştan `adet` tane örnek demek.

In [ ]:
adetler = [18, 45, 90, 135, 180]
dogruluklar = []
for adet in adetler:
    yeni_model = KNeighborsClassifier(n_neighbors=5)
    yeni_model.fit(X_egitim[:adet], y_egitim[:adet])
    dogruluk = yeni_model.score(X_test, y_test)
    print(adet, round(dogruluk, 2))
    dogruluklar.append(dogruluk)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(adetler, dogruluklar, marker="o")
plt.xlabel("Eğitim örneği sayısı")
plt.ylabel("Test doğruluğu")
plt.show()

Eğri hep yükseliyor mu? Ortada düşüyorsa düzeltmeye çalışma; ödevde konuşacağız.

## Adım 7 — Model ne öğrendi? Kafe paletini sor

Bu kez ölçmüyoruz, sadece soru soruyoruz: modeli **tüm veriyle** eğitiyoruz.

Konu 1'deki "sessiz çalışma kafesi" için aday bir palet. Renk seçicideki RGB sayılarıyla:
bej `#E8DCC4`, sütlü kahve `#D4A373`, adaçayı `#A3B18A`, orman yeşili `#344E41`, kirli beyaz `#F5F5F5`.

In [ ]:
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X, y)

bej = [232, 220, 196]
sutlu_kahve = [212, 163, 115]
adacayi = [163, 177, 138]
orman_yesili = [52, 78, 65]
kirli_beyaz = [245, 245, 245]

palet = [bej, sutlu_kahve, adacayi, orman_yesili, kirli_beyaz]
tahminler = model.predict(palet)
print(tahminler)

Cevaplar paletle aynı sırada: ilki bej, ikincisi sütlü kahve… Çıktıda virgül yok; scikit-learn
kendi liste türünü kullanıyor, okuması aynı. Kafe paleti "sakin" mi çıktı?

## Adım 8 — Hiç görmediği bir renk

Adım 3'teki testte her renk eğitimde de vardı: model testteki kırık beyazın "kardeşlerini" görmüştü.
Şimdi kırık beyazı **tamamen** dışarıda bırakan bir model kuralım.

In [ ]:
X_haric = []
y_haric = []
for kayit in kayitlar:
    if kayit["ad"] != "kırık beyaz":
        r = int(kayit["r"])
        g = int(kayit["g"])
        b = int(kayit["b"])
        X_haric.append([r, g, b])
        y_haric.append(kayit["etiket"])

print(len(X_haric))

In [ ]:
model_haric = KNeighborsClassifier(n_neighbors=5)
model_haric.fit(X_haric, y_haric)

kirik_beyaz = [241, 250, 238]
tahminler = model_haric.predict([kirik_beyaz])
print(tahminler[0])

Adım 7'deki model kırık beyazı görmüştü. O ne diyor?

In [ ]:
tahminler = model.predict([kirik_beyaz])
print(tahminler[0])

Sınıfın çoğu kırık beyaza ne demişti (Adım 1)? `"kırık beyaz"` yerine `"gri mavi"` yaz,
sayıları `[141, 153, 174]` yap ve Adım 8'in üç hücresini sırayla yeniden çalıştır.

## Bonus — Kendi rengini sor

Renk seçiciden bir renk seç, RGB sayılarını yaz. Modelle aynı fikirde misin?

In [ ]:
benim_rengim = [255, 0, 0]
tahminler = model.predict([benim_rengim])
print(tahminler[0])